In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("dataset_clinica_20261.csv", encoding="utf-8")

def corrigir_mojibake(valor):
    if isinstance(valor, str) and ("Ã" in valor or "Â" in valor):
        try:
            return valor.encode("latin1").decode("utf-8")
        except UnicodeError:
            return valor
    return valor

colunas_texto = df.select_dtypes(include="object").columns
df[colunas_texto] = df[colunas_texto].apply(lambda col: col.map(corrigir_mojibake))

df.head()

,cd_processo,id_processo,classe,assunto,magistrado,comarca,foro,vara,data_disponibilizacao,decisao
0,5B0005D030000,1002017-64.2024.8.26.0191,Procedimento do Juizado Especial Cível,Bancários,LUCIANA DO CARMO NOGUEIRA,Ferraz de Vasconcelos,Foro de Ferraz de Vasconcelos,Vara do Juizado Especial Cível e Criminal,27/08/2024,SENTENÇA\n\nProcesso Digital Nº:\t1002017-64.2...
1,03001EPSV0000,1034196-67.2023.8.26.0003,Procedimento Comum Cível,Bancários,Laura Mota Lima de Oliveira Baccin,SÃO PAULO,Foro Regional III - Jabaquara,1ª Vara Cível,27/08/2024,SENTENÇA\n\nProcesso Digital nº:\t1034196-67.2...
2,I20002VT10000,1000247-51.2023.8.26.0650,Procedimento Comum Cível,Empréstimo consignado,Marcia Yoshie Ishikawa,Valinhos,Foro de Valinhos,3ª Vara,27/08/2024,SENTENÇA\n\nProcesso Digital nº:\t1000247-51.2...
3,DE000DZ0N0000,1010110-16.2024.8.26.0482,Procedimento Comum Cível,Bancários,Leonardo Mazzilli Marcondes,Presidente Prudente,Foro de Presidente Prudente,4ª Vara Cível,27/08/2024,SENTENÇA\n\nProcesso Digital nº:\t1010110-16.2...
4,2S001VMCT0000,1101723-02.2024.8.26.0100,Procedimento Comum Cível,Bancários,MICHELLE FABIOLA DITTERT PUPULIM,SÃO PAULO,Foro Regional III - Jabaquara,6ª Vara Cível,27/08/2024,SENTENÇA\n\nProcesso Digital nº:\t1101723-02.2...


In [3]:
df.columns

Index(['cd_processo', 'id_processo', 'classe', 'assunto', 'magistrado',
       'comarca', 'foro', 'vara', 'data_disponibilizacao', 'decisao'],
      dtype='object')

In [4]:
# Padroniza os nomes de magistrado em maiúsculas
df["magistrado"] = df["magistrado"].astype(str).str.strip().str.upper()

# Defina um nome para filtrar (ex.: "LUCIANA DO CARMO NOGUEIRA") ou deixe None
magistrado_filtro = None

if magistrado_filtro:
    df_filtrado = df[df["magistrado"].str.contains(magistrado_filtro.upper(), na=False)]
else:
    df_filtrado = df

coluna_assunto = "assunto" if "assunto" in df_filtrado.columns else "assuntos"
colunas_para_analisar = ["classe", coluna_assunto, "foro", "vara", "magistrado"]

for coluna in colunas_para_analisar:
    print(f"\n=== value_counts: {coluna} ===")
    print(df_filtrado[coluna].value_counts(dropna=False))


=== value_counts: classe ===
classe
Procedimento Comum Cível                                          16652
Procedimento do Juizado Especial Cível                             4053
Cumprimento de sentença                                            1766
Cumprimento Provisório de Sentença                                   99
Ação de Exigir Contas                                                84
Produção Antecipada da Prova                                         50
Procedimento de Repactuação de Dívidas (Superendividamento)          33
Tutela Antecipada Antecedente                                        29
Tutela Cautelar Antecedente                                          15
Cumprimento Provisório de Decisão                                    14
Liquidação de Sentença pelo Procedimento Comum                       13
Embargos à Execução                                                  12
NaN                                                                  10
Liquidação por Arbitramento

In [5]:
coluna_decisao = "decisao"
coluna_processo = "id_processo" if "id_processo" in df.columns else None

top10 = df[[coluna_decisao]].head(10).copy()
if coluna_processo:
    top10.insert(0, coluna_processo, df[coluna_processo].head(10).values)

linhas = []
for i, (_, row) in enumerate(top10.iterrows(), start=1):
    if coluna_processo:
        linhas.append(f"Processo {i} - {row[coluna_processo]}")
    else:
        linhas.append(f"Processo {i}")
    linhas.append(str(row[coluna_decisao]))
    linhas.append("-" * 80)

texto_saida = "\n".join(linhas)
arquivo_saida = "top10_decisoes.txt"

with open(arquivo_saida, "w", encoding="utf-8") as f:
    f.write(texto_saida)

print(f"Arquivo salvo: {arquivo_saida}")

Arquivo salvo: top10_decisoes.txt


In [6]:
import re
import unicodedata

def normalizar_texto(texto):
    if pd.isna(texto):
        return ""
    texto = str(texto).lower()
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("ascii")
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

padrao_justica_gratuita = re.compile(
    r"\b("
    r"justica\s+gratuita|"
    r"gratuidade\s+da\s+justica|"
    r"beneficio\s+da\s+justica\s+gratuita|"
    r"assistencia\s+judiciaria\s+gratuita|"
    r"ajg"
    r")\b",
    flags=re.IGNORECASE,
 )

coluna_flag = "tem_justica_gratuita"
mascara_justica_gratuita = df["decisao"].apply(
    lambda x: bool(padrao_justica_gratuita.search(normalizar_texto(x)))
 )

if coluna_flag in df.columns:
    df[coluna_flag] = mascara_justica_gratuita
else:
    posicao = df.columns.get_loc("magistrado") + 1 if "magistrado" in df.columns else len(df.columns)
    df.insert(posicao, coluna_flag, mascara_justica_gratuita)

df[["magistrado", coluna_flag, "decisao"]].head(10)

,magistrado,tem_justica_gratuita,decisao
0,LUCIANA DO CARMO NOGUEIRA,True,SENTENÇA\n\nProcesso Digital Nº:\t1002017-64.2...
1,LAURA MOTA LIMA DE OLIVEIRA BACCIN,True,SENTENÇA\n\nProcesso Digital nº:\t1034196-67.2...
2,MARCIA YOSHIE ISHIKAWA,False,SENTENÇA\n\nProcesso Digital nº:\t1000247-51.2...
3,LEONARDO MAZZILLI MARCONDES,True,SENTENÇA\n\nProcesso Digital nº:\t1010110-16.2...
4,MICHELLE FABIOLA DITTERT PUPULIM,False,SENTENÇA\n\nProcesso Digital nº:\t1101723-02.2...
5,ALÉSSIO MARTINS GONÇALVES,False,SENTENÇA\n\nProcesso nº:\t1009437-72.2023.8.26...
6,VINICIUS RODRIGUES VIEIRA,False,Processo nº:\t0067587-64.2009.8.26.0506\nClass...
7,JOANNA TERRA SAMPAIO DOS SANTOS,False,SENTENÇA\n\nProcesso nº:\t1023563-21.2024.8.26...
8,MÔNICA DE CASSIA THOMAZ PEREZ REIS LOBO,False,SENTENÇA\n\nProcesso nº:\t1101899-78.2024.8.26...
9,MELISSA BERTOLUCCI,False,SENTENÇA\n\nProcesso Digital nº:\t1120452-76.2...


In [7]:
print(df_filtrado["tem_justica_gratuita"].value_counts(dropna=False))

tem_justica_gratuita
True     12434
False    10432
Name: count, dtype: int64


In [8]:
coluna_assunto = "assunto" if "assunto" in df.columns else "assuntos"
coluna_flag = "tem_justica_gratuita"

# Tabela por assunto x justiça gratuita
resumo_assunto = pd.crosstab(df[coluna_assunto], df[coluna_flag], dropna=False)
resumo_assunto = resumo_assunto.rename(columns={False: "nao", True: "sim"})

# Garante as colunas mesmo se uma categoria não aparecer
for c in ["sim", "nao"]:
    if c not in resumo_assunto.columns:
        resumo_assunto[c] = 0

resumo_assunto["total"] = resumo_assunto["sim"] + resumo_assunto["nao"]
resumo_assunto["pct_sim"] = (resumo_assunto["sim"] / resumo_assunto["total"] * 100).round(2)
resumo_assunto = resumo_assunto.sort_values(["sim", "total"], ascending=False)

print("Resumo por assunto (justiça gratuita: sim/nao):")
display(resumo_assunto.head(30))

print("\nAssuntos com pelo menos 1 ocorrência de JUSTIÇA GRATUITA:")
display(resumo_assunto[resumo_assunto["sim"] > 0].head(30))

print("\nAssuntos sem ocorrência de JUSTIÇA GRATUITA:")
display(resumo_assunto[resumo_assunto["sim"] == 0].head(30))

Resumo por assunto (justiça gratuita: sim/nao):


tem_justica_gratuita,nao,sim,total,pct_sim
assunto,,,,
Bancários,8502,8872,17374,51.06
Empréstimo consignado,1435,2969,4404,67.42
"Revisão de Juros Remuneratórios, Capitalização/Anatocismo",273,336,609,55.17
Crédito Direto ao Consumidor - CDC,108,128,236,54.24
Tarifas,102,121,223,54.26
Crédito Rotativo,12,8,20,40.00



Assuntos com pelo menos 1 ocorrência de JUSTIÇA GRATUITA:


tem_justica_gratuita,nao,sim,total,pct_sim
assunto,,,,
Bancários,8502,8872,17374,51.06
Empréstimo consignado,1435,2969,4404,67.42
"Revisão de Juros Remuneratórios, Capitalização/Anatocismo",273,336,609,55.17
Crédito Direto ao Consumidor - CDC,108,128,236,54.24
Tarifas,102,121,223,54.26
Crédito Rotativo,12,8,20,40.00



Assuntos sem ocorrência de JUSTIÇA GRATUITA:


tem_justica_gratuita,nao,sim,total,pct_sim
assunto,,,,


In [9]:
coluna_assunto = "assunto" if "assunto" in df.columns else "assuntos"
coluna_flag = "tem_justica_gratuita"

# Tabela cruzada normalizada pelas colunas (sim/nao)
# Isso calculará a proporção de cada assunto DENTRO do universo dos que pediram (sim) e dos que não pediram (nao)
resumo_assunto_norm = pd.crosstab(df[coluna_assunto], df[coluna_flag], normalize='columns', dropna=False)
resumo_assunto_norm = resumo_assunto_norm.rename(columns={False: "pct_nao", True: "pct_sim"})

# Converte proporção para porcentagem
resumo_assunto_norm = (resumo_assunto_norm * 100).round(2)

# Garante as colunas caso o dataset fosse 100% sim ou não
for c in ["pct_sim", "pct_nao"]:
    if c not in resumo_assunto_norm.columns:
        resumo_assunto_norm[c] = 0.0

# Calcula a diferença percentual entre os dois grupos
resumo_assunto_norm["diferenca_pct (sim - nao)"] = resumo_assunto_norm["pct_sim"] - resumo_assunto_norm["pct_nao"]

resumo_assunto_norm = resumo_assunto_norm.sort_values("pct_sim", ascending=False)

print("Percentual de cada assunto dentro de cada grupo (Justiça Gratuita: SIM vs NÃO):")
display(resumo_assunto_norm.head(30))

print("\nAssuntos com maior prevalência NO GRUPO SIM em comparação ao GRUPO NÃO (maior assimetria para o sim):")
display(resumo_assunto_norm.sort_values("diferenca_pct (sim - nao)", ascending=False).head(20))

print("\nAssuntos com maior prevalência NO GRUPO NÃO em comparação ao GRUPO SIM:")
display(resumo_assunto_norm.sort_values("diferenca_pct (sim - nao)", ascending=True).head(20))

Percentual de cada assunto dentro de cada grupo (Justiça Gratuita: SIM vs NÃO):


tem_justica_gratuita,pct_nao,pct_sim,diferenca_pct (sim - nao)
assunto,,,
Bancários,81.50,71.35,-10.15
Empréstimo consignado,13.76,23.88,10.12
"Revisão de Juros Remuneratórios, Capitalização/Anatocismo",2.62,2.70,0.08
Crédito Direto ao Consumidor - CDC,1.04,1.03,-0.01
Tarifas,0.98,0.97,-0.01
Crédito Rotativo,0.12,0.06,-0.06



Assuntos com maior prevalência NO GRUPO SIM em comparação ao GRUPO NÃO (maior assimetria para o sim):


tem_justica_gratuita,pct_nao,pct_sim,diferenca_pct (sim - nao)
assunto,,,
Empréstimo consignado,13.76,23.88,10.12
"Revisão de Juros Remuneratórios, Capitalização/Anatocismo",2.62,2.70,0.08
Tarifas,0.98,0.97,-0.01
Crédito Direto ao Consumidor - CDC,1.04,1.03,-0.01
Crédito Rotativo,0.12,0.06,-0.06
Bancários,81.50,71.35,-10.15



Assuntos com maior prevalência NO GRUPO NÃO em comparação ao GRUPO SIM:


tem_justica_gratuita,pct_nao,pct_sim,diferenca_pct (sim - nao)
assunto,,,
Bancários,81.50,71.35,-10.15
Crédito Rotativo,0.12,0.06,-0.06
Tarifas,0.98,0.97,-0.01
Crédito Direto ao Consumidor - CDC,1.04,1.03,-0.01
"Revisão de Juros Remuneratórios, Capitalização/Anatocismo",2.62,2.70,0.08
Empréstimo consignado,13.76,23.88,10.12


In [10]:
# Padrão regex para capturar o resultado da sentença
# Procura conjugações de "julgar" seguido (até 80 caracteres de distância) do resultado
padrao_resultado = re.compile(
    r"\b(?:julgo|julga|julgam|julgado|julgada|julgar|julgo-o)\b" # Variações do verbo julgar
    r".{0,80}?" # Permite algumas palavras no meio (ex: 'o pedido inicial', 'liminarmente', 'totalmente', etc)
    r"\b(parcialmente procedente|parcialmente improcedente|totalmente procedente|totalmente improcedente|procedente|improcedente|extinto)\b",
    flags=re.IGNORECASE
)

def extrair_resultado(texto):
    if pd.isna(texto):
        return "nao_identificado"
    
    # normalizar_texto foi definida na célula de justiça gratuita
    texto_norm = normalizar_texto(texto) 
    
    match = padrao_resultado.search(texto_norm)
    if match:
        # Pega a palavra/expressão capturada e troca espaço por underline
        resultado = match.group(1).strip().replace(" ", "_").lower()
        return resultado
        
    return "nao_identificado"

coluna_decisao = "decisao"
coluna_resultado = "resultado_sentenca"

if coluna_decisao in df.columns:
    # Aplica a função para criar a nova coluna no DataFrame
    df[coluna_resultado] = df[coluna_decisao].apply(extrair_resultado)
    
    print("Distribuição dos resultados extraídos:")
    print(df[coluna_resultado].value_counts(dropna=False))
    
    print("\nAmostra de decisões e os resultados extraídos:")
    # Filtra alguns casos onde identificou algo para mostrar de exemplo
    df_exemplo = df[df[coluna_resultado] != "nao_identificado"]
    if not df_exemplo.empty:
        display(df_exemplo[[coluna_decisao, coluna_resultado]].head(10))
    else:
        display(df[[coluna_decisao, coluna_resultado]].head(10))
else:
    print(f"Coluna '{coluna_decisao}' não encontrada para aplicar a extração.")

Distribuição dos resultados extraídos:
resultado_sentenca
nao_identificado             7336
extinto                      6797
improcedente                 4625
procedente                   2092
parcialmente_procedente      1955
totalmente_procedente          36
totalmente_improcedente        21
parcialmente_improcedente       4
Name: count, dtype: int64

Amostra de decisões e os resultados extraídos:


,decisao,resultado_sentenca
0,SENTENÇA\n\nProcesso Digital Nº:\t1002017-64.2...,extinto
1,SENTENÇA\n\nProcesso Digital nº:\t1034196-67.2...,improcedente
3,SENTENÇA\n\nProcesso Digital nº:\t1010110-16.2...,procedente
5,SENTENÇA\n\nProcesso nº:\t1009437-72.2023.8.26...,parcialmente_procedente
6,Processo nº:\t0067587-64.2009.8.26.0506\nClass...,extinto
7,SENTENÇA\n\nProcesso nº:\t1023563-21.2024.8.26...,extinto
9,SENTENÇA\n\nProcesso Digital nº:\t1120452-76.2...,extinto
10,SENTENÇA\nProcesso nº:\t1011520-08.2024.8.26.0...,extinto
11,SENTENÇA\n\nProcesso Digital nº:\t0014038-08.2...,extinto
12,SENTENÇA\n\nProcesso Digital nº:\t1057594-09.2...,extinto
